# Replication: "Smart Green Nudging" — von Zahn et al. (2024)
> *Marketing Science* — https://pubsonline.informs.org/doi/10.1287/mksc.2022.0393

**Sections**
1. Setup & Data Loading
2. Digital Footprint Feature Engineering
3. ATE Estimation (OLS · IPW · AIPW)
4. Robustness Checks (permutation test · bandwidth sensitivity)
5. Causal Forest — Heterogeneous Treatment Effects (CATE)
6. Nudge Targeting Policy & Profit Analysis
7. Summary & Conclusions
8. Ad Hoc Analysis & Extensions
   - 8.1 Business / Retail Strategy — Profit Levers & Targeting Decisions
   - 8.2 Technical ML — Adapting Causal Forests to Other Domains
   - 8.3 Cross-Industry Extensions

All heavy logic lives in `src/` — each replication cell is a one-liner import + call.

---
> **Note:** This replication uses synthetic data generated to match the paper's
> described data-generating process. The original retailer data is proprietary.

## 0. Install Dependencies
```bash
pip install econml pandas numpy matplotlib seaborn scikit-learn statsmodels
```

---
## 1. Setup & Data Loading

In [ ]:
import sys, warnings
warnings.filterwarnings('ignore')
sys.path.insert(0, 'src')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
from pathlib import Path

from features      import engineer_features, get_feature_cols, check_covariate_balance
from ate           import run_ate_suite, permutation_test, bandwidth_sensitivity
from causal_forest import fit_causal_forest, compute_rate
from plots         import *

RANDOM_SEED   = 42
TREATMENT_COL = 'treatment'
OUTCOME_COL   = 'returned'
np.random.seed(RANDOM_SEED)

In [ ]:
DATA_DIR = Path('data')
df_train = pd.read_csv(DATA_DIR / 'train.csv')
df_test  = pd.read_csv(DATA_DIR / 'test.csv')

print(f'Train: {df_train.shape}  |  Test: {df_test.shape}')
df_train.head()

In [ ]:
plot_descriptive_overview(df_train, TREATMENT_COL, OUTCOME_COL);

---
## 2. Digital Footprint Feature Engineering

The paper constructs behavioural features from customers' digital footprints —
browsing sessions, past return history, basket composition, and device usage.
Feature logic is in `src/features.py`.

In [ ]:
df_train = engineer_features(df_train)
df_test  = engineer_features(df_test)

FEATURE_COLS = get_feature_cols(df_train, exclude=[TREATMENT_COL, OUTCOME_COL])
print(f'{len(FEATURE_COLS)} numeric features after engineering:')
print(FEATURE_COLS)
df_train[FEATURE_COLS].describe().round(3)

In [ ]:
plot_correlation_heatmap(df_train, FEATURE_COLS, OUTCOME_COL);

In [ ]:
balance = check_covariate_balance(df_train, TREATMENT_COL, FEATURE_COLS)
print(balance.to_string(index=False))
plot_covariate_balance(balance);

---
## 3. ATE Estimation — Field Experiment

We estimate the **Average Treatment Effect (ATE)** using four estimators:
OLS (naive), OLS + covariates (HC3), IPW (Horvitz-Thompson), and AIPW
(doubly-robust). All logic is in `src/ate.py`.

In [ ]:
X_train = df_train[FEATURE_COLS].fillna(0)
T_train = df_train[TREATMENT_COL]
Y_train = df_train[OUTCOME_COL]

X_test  = df_test[FEATURE_COLS].fillna(0)
T_test  = df_test[TREATMENT_COL]
Y_test  = df_test[OUTCOME_COL]

ate_results = run_ate_suite(Y_train, T_train, X_train, n_boot=500, seed=RANDOM_SEED)
print(ate_results.summary())

In [ ]:
plot_ate_forest(ate_results.to_dataframe());

---
## 4. Robustness Checks

### 4.1 Permutation / Placebo Test

In [ ]:
perm = permutation_test(Y_train, T_train, X_train, n_permutations=1_000, seed=RANDOM_SEED)
print(f"Observed ATE : {perm['observed_ate']:+.4f}")
print(f"Null mean    : {perm['null_mean']:+.4f}  std={perm['null_std']:.4f}")
print(f"Empirical p  : {perm['p_value']:.4f}")
plot_permutation_null(perm);

### 4.2 Bandwidth / Sample-Restriction Sensitivity

In [ ]:
bw_df = bandwidth_sensitivity(
    df_train, Y_col=OUTCOME_COL, T_col=TREATMENT_COL, feature_cols=FEATURE_COLS
)
print(bw_df[['bandwidth','n','ate','ci_lo','ci_hi','pvalue','sig']].to_string(index=False))
plot_bandwidth_sensitivity(bw_df);

---
## 5. Causal Forest — Heterogeneous Treatment Effects (CATE)

In [ ]:
cf = fit_causal_forest(
    Y_train, T_train, X_train, X_test,
    feature_names=FEATURE_COLS,
    n_estimators=2_000,
    seed=RANDOM_SEED,
)
print(cf.summary())

In [ ]:
df_test = df_test.copy()
df_test['cate']    = cf.cate
df_test['cate_lb'] = cf.cate_lb
df_test['cate_ub'] = cf.cate_ub
pct_benefit = (cf.cate < 0).mean()

plot_cate_distribution(cf);

In [ ]:
plot_feature_importance(cf.feat_imp);

In [ ]:
rate_res = compute_rate(cf.cate, Y_test.values, T_test.values)
print(f"RATE = {rate_res['rate']:+.4f}  (positive → smart targeting beats random)")
plot_toc_curve(rate_res);

---
## 6. Nudge Targeting Policy & Profit Analysis

In [ ]:
RETURN_COST = 15.0
NUDGE_COST  = 0.10
ATE_CTRL    = ate_results.ate[1]

df_pol = df_test.sort_values('cate').reset_index(drop=True)
n      = len(df_pol)
fracs  = np.linspace(0, 1, 101)

profit_smart, profit_univ = [], []
for frac in fracs:
    k = int(frac * n)
    profit_smart.append((-df_pol['cate'].iloc[:k].clip(upper=0)).sum() * RETURN_COST - k * NUDGE_COST)
    profit_univ.append(frac * n * max(0, -ATE_CTRL) * RETURN_COST - k * NUDGE_COST)

profit_smart = np.array(profit_smart)
profit_univ  = np.array(profit_univ)
best_frac    = fracs[np.argmax(profit_smart)]

print(f'Optimal targeting share : {best_frac:.1%}')
print(f'Max profit — smart      : €{profit_smart.max():.2f}')
print(f'Max profit — universal  : €{profit_univ.max():.2f}')
print(f'Value of personalisation: €{profit_smart.max() - profit_univ.max():.2f}')
plot_policy_curve(fracs, profit_smart, profit_univ, best_frac);

In [ ]:
df_test['cate_quartile'] = pd.qcut(
    df_test['cate'], q=4,
    labels=['Q1\n(most\nreceptive)', 'Q2', 'Q3', 'Q4\n(least\nreceptive)']
)
seg = df_test.groupby('cate_quartile', observed=True).agg(
    n=          (OUTCOME_COL, 'count'),
    return_rate=(OUTCOME_COL, 'mean'),
    mean_cate=  ('cate',      'mean'),
).round(4)
print(seg)
plot_cate_segments(seg);

---
## 7. Summary & Conclusions

In [ ]:
summary = pd.DataFrame({
    'Metric': [
        'ATE — OLS naive',
        'ATE — OLS + controls',
        'ATE — IPW',
        'ATE — AIPW (doubly-robust)',
        'Permutation p-value',
        'Mean CATE (causal forest)',
        'CATE std (heterogeneity)',
        '% customers CATE < 0',
        'Confidence score',
        'RATE',
        'Optimal targeting share',
        'Max profit — smart targeting',
        'Max profit — universal nudging',
        'Value of personalisation',
    ],
    'Value': [
        f"{ate_results.ate[0]:+.4f}  (p={ate_results.pvalue[0]:.4f})",
        f"{ate_results.ate[1]:+.4f}  (p={ate_results.pvalue[1]:.4f})",
        f"{ate_results.ate[2]:+.4f}  (p={ate_results.pvalue[2]:.4f})",
        f"{ate_results.ate[3]:+.4f}  (p={ate_results.pvalue[3]:.4f})",
        f"{perm['p_value']:.4f}",
        f"{cf.cate.mean():+.4f}  ({cf.cate.mean()*100:+.2f} pp)",
        f"{cf.cate.std():.4f}",
        f"{pct_benefit:.1%}",
        f"{cf.conf_score:.1%}",
        f"{rate_res['rate']:+.4f}",
        f"{best_frac:.1%}",
        f"€{profit_smart.max():.2f}",
        f"€{profit_univ.max():.2f}",
        f"€{profit_smart.max() - profit_univ.max():.2f}",
    ]
})
print('═' * 65)
print('  REPLICATION SUMMARY')
print('═' * 65)
print(summary.to_string(index=False))
print('═' * 65)

---
---
# 8. Ad Hoc Analysis & Extensions

The sections below move beyond the replication itself and ask: **what is this methodology actually good for, and where else could it go?** Each section pairs a formal academic discussion with an accessible plain-language summary, followed by illustrative code.

---
## 8.1 Business / Retail Strategy — Profit Levers & Targeting Decisions

### Academic Discussion

The core managerial contribution of von Zahn et al. (2024) lies not in the average treatment effect — which is modest in absolute terms — but in the **heterogeneity of that effect across the customer distribution**. Under a universal nudging policy, the firm incurs the cost of the intervention for every customer, including those for whom the nudge is ineffective or counterproductive (i.e., CATE ≥ 0). The causal forest enables a **data-driven targeting rule** that limits nudge exposure to the subset of customers for whom the expected profit gain exceeds the intervention cost, formally: nudge customer $i$ iff $-\hat{\tau}_i \cdot c_{\text{return}} > c_{\text{nudge}}$, where $\hat{\tau}_i$ is the estimated CATE, $c_{\text{return}}$ is the marginal cost of a returned item, and $c_{\text{nudge}}$ is the cost per nudge delivered. This framework is directly analogous to the **Neyman-Pearson classification** literature (Tian et al., 2014) and the optimal policy tree work of Athey & Wager (2021). Critically, the value of personalisation is sensitive to three levers: the dispersion of the CATE distribution, the cost ratio $c_{\text{nudge}} / c_{\text{return}}$, and the share of customers with a genuinely negative CATE.

**Plain-language summary:**
- Not every customer responds to a green nudge — some ignore it, some are already not returning items, and a few might even be annoyed by it.
- The causal forest figures out *which* customers are actually worth nudging by estimating an individual "responsiveness score" (CATE) for everyone.
- The retailer should only show the nudge when the expected savings from fewer returns outweigh the small cost of sending the nudge.
- Three things determine how much money this saves vs. nudging everyone: (1) how spread out the responsiveness scores are, (2) how expensive returns are relative to nudges, and (3) what fraction of customers actually respond.
- The code below lets you stress-test this by changing the cost assumptions.

In [ ]:
# ── Extension 8.1: Profit lever sensitivity sweep ─────────────────────────────
# How does the value of smart targeting change as return costs and nudge costs vary?

return_costs = [5, 10, 15, 25, 40]   # € per returned item
nudge_costs  = [0.05, 0.10, 0.25, 0.50]  # € per nudge sent

results_grid = []
for rc in return_costs:
    for nc in nudge_costs:
        p_smart, p_univ = [], []
        for frac in fracs:
            k = int(frac * n)
            p_smart.append((-df_pol['cate'].iloc[:k].clip(upper=0)).sum() * rc - k * nc)
            p_univ.append(frac * n * max(0, -ATE_CTRL) * rc - k * nc)
        p_smart = np.array(p_smart)
        p_univ  = np.array(p_univ)
        results_grid.append({
            'return_cost': rc,
            'nudge_cost':  nc,
            'smart_gain':  p_smart.max(),
            'univ_gain':   p_univ.max(),
            'personalisation_value': p_smart.max() - p_univ.max(),
            'optimal_frac': fracs[np.argmax(p_smart)],
        })

grid_df = pd.DataFrame(results_grid)

# Heatmap: value of personalisation across cost combinations
pivot = grid_df.pivot(index='return_cost', columns='nudge_cost', values='personalisation_value')
fig, ax = plt.subplots(figsize=(8, 5))
sns.heatmap(pivot, annot=True, fmt='.0f', cmap='YlGn', ax=ax,
            cbar_kws={'label': '€ gain from smart vs. universal nudging'})
ax.set_title('Value of Personalisation (€) by Return Cost vs. Nudge Cost')
ax.set_xlabel('Nudge cost per customer (€)')
ax.set_ylabel('Return cost per item (€)')
plt.tight_layout()
plt.show()

print("\nFull grid:")
print(grid_df.round(2).to_string(index=False))

In [ ]:
# ── Extension 8.1b: Break-even analysis ───────────────────────────────────────
# At what return cost does smart targeting start beating universal nudging?
# At what nudge cost does it stop being worth nudging anyone?

rc_sweep = np.linspace(1, 50, 100)
smart_maxes = []
univ_maxes  = []

for rc in rc_sweep:
    ps = [(-df_pol['cate'].iloc[:int(f*n)].clip(upper=0)).sum() * rc - int(f*n) * NUDGE_COST
          for f in fracs]
    pu = [f * n * max(0, -ATE_CTRL) * rc - int(f*n) * NUDGE_COST for f in fracs]
    smart_maxes.append(max(ps))
    univ_maxes.append(max(pu))

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(rc_sweep, smart_maxes, color='#4CAF82', linewidth=2.5, label='Smart targeting')
ax.plot(rc_sweep, univ_maxes,  color='#5B8DB8', linewidth=2.5, linestyle='--', label='Universal nudging')
ax.axhline(0, color='black', linestyle=':', linewidth=1, alpha=0.5)
ax.fill_between(rc_sweep,
                smart_maxes, univ_maxes,
                where=[s > u for s, u in zip(smart_maxes, univ_maxes)],
                alpha=0.15, color='#4CAF82', label='Smart outperforms')
ax.set_xlabel('Return cost per item (€)')
ax.set_ylabel('Max achievable profit gain (€)')
ax.set_title(f'Break-Even Analysis: Smart vs. Universal Nudging\n(nudge cost fixed at €{NUDGE_COST})')
ax.legend()
plt.tight_layout()
plt.show()

---
## 8.2 Technical ML — Adapting Causal Forests to Other Domains

### Academic Discussion

The Causal Forest / Double Machine Learning framework (Chernozhukov et al., 2018; Wager & Athey, 2018) is domain-agnostic: it requires only a binary or continuous treatment $T$, an outcome $Y$, and a covariate matrix $X$. The key modelling choices that practitioners must revisit when applying this to a new domain are: (1) **the nuisance model** — the learners for $E[Y|X]$ and $E[T|X]$ should be appropriate for the marginal distributions of $Y$ and $T$ in the new domain (e.g., a survival model for time-to-event outcomes, or a Poisson model for count data); (2) **the honesty and subsampling regime** — the `min_samples_leaf` and `subsample_fraction` hyperparameters control the bias-variance tradeoff of the CATE estimates and should be tuned via cross-validation on a proper scoring rule; and (3) **the estimand of interest** — the framework naturally extends to the ATT (average treatment effect on the treated), LATE (local ATE for compliers in an IV setting), or the policy-optimal ATE under budget constraints. Transferred to domains with observational (non-experimental) data, the identifying assumption shifts from random assignment to **strong ignorability** (unconfoundedness), which requires that all confounders are observed — a much stronger and often untestable claim.

**Plain-language summary:**
- The causal forest is not specific to green nudges or e-commerce. It's a general tool for answering: *"does this intervention work, and for whom?"*
- To use it in a new setting, you mainly need to: (a) define what your treatment, outcome, and customer features are, and (b) choose appropriate prediction models for your data type.
- It works best when the treatment was randomly assigned (like in an A/B test). With observational data, you need to assume you've measured every variable that simultaneously affects both the treatment and the outcome — which is often hard to guarantee.
- The table below maps out how the framework translates to four other domains.

In [ ]:
# ── Extension 8.2: Domain translation table ───────────────────────────────────

domain_table = pd.DataFrame([
    {
        'Domain':        'E-commerce returns (original)',
        'Treatment T':   'Green nudge shown (binary)',
        'Outcome Y':     'Item returned (binary)',
        'Key features X':'Past return rate, order value, session duration',
        'Nuisance model':'GBM classifier / GBM regressor',
        'Main assumption':'RCT (randomisation)',
    },
    {
        'Domain':        'Healthcare — medication adherence',
        'Treatment T':   'SMS reminder sent (binary)',
        'Outcome Y':     'Days adherent in next 30d (count)',
        'Key features X':'Age, diagnosis, prior adherence, income proxy',
        'Nuisance model':'Poisson / GBM',
        'Main assumption':'Unconfoundedness (observational)',
    },
    {
        'Domain':        'EdTech — online course completion',
        'Treatment T':   'Personalised study plan (binary)',
        'Outcome Y':     'Course completed (binary)',
        'Key features X':'Prior quiz scores, login frequency, device',
        'Nuisance model':'Logistic / GBM classifier',
        'Main assumption':'RCT (A/B test on platform)',
    },
    {
        'Domain':        'Credit — loan default prevention',
        'Treatment T':   'Early intervention call (binary)',
        'Outcome Y':     'Default within 90 days (binary)',
        'Key features X':'Credit score, payment history, loan-to-value',
        'Nuisance model':'GBM classifier',
        'Main assumption':'Unconfoundedness (if calls not targeted)',
    },
    {
        'Domain':        'Energy — smart thermostat nudge',
        'Treatment T':   'Peak-hour alert sent (binary)',
        'Outcome Y':     'kWh consumed in peak window (continuous)',
        'Key features X':'House size, past consumption, outdoor temp',
        'Nuisance model':'GBM regressor',
        'Main assumption':'RCT (randomised rollout)',
    },
])

print(domain_table.to_string(index=False))

In [ ]:
# ── Extension 8.2b: Simulate CATE heterogeneity under different DGPs ──────────
# This cell illustrates how the shape of the CATE distribution changes
# depending on how heterogeneous the true treatment effects are.
# Useful for understanding when a causal forest adds the most value.

rng = np.random.default_rng(RANDOM_SEED)
n_sim = 2_000

scenarios = {
    'Homogeneous\n(no heterogeneity)':  rng.normal(-0.05, 0.01, n_sim),
    'Low heterogeneity':                rng.normal(-0.05, 0.03, n_sim),
    'Moderate heterogeneity\n(paper)':  rng.normal(-0.05, 0.07, n_sim),
    'High heterogeneity':               rng.normal(-0.03, 0.15, n_sim),
}

fig, axes = plt.subplots(1, 4, figsize=(15, 3.5), sharey=False)
colors = ['#5B8DB8', '#4CAF82', '#E07B54', '#9B59B6']

for ax, (label, cates), color in zip(axes, scenarios.items(), colors):
    ax.hist(cates, bins=40, color=color, edgecolor='white', alpha=0.85)
    ax.axvline(0,          color='black', linestyle='-',  linewidth=0.8, alpha=0.5)
    ax.axvline(cates.mean(), color='red', linestyle='--', linewidth=1.5,
               label=f'Mean={cates.mean():.3f}')
    pct_pos = (cates < 0).mean()
    ax.set_title(f'{label}\n{pct_pos:.0%} benefit', fontsize=9)
    ax.set_xlabel('CATE')
    ax.xaxis.set_major_formatter(mtick.PercentFormatter(1.0, decimals=0))
    ax.legend(fontsize=7)

axes[0].set_ylabel('Count')
fig.suptitle('Simulated CATE Distributions: When Does a Causal Forest Add Value?',
             fontsize=11, y=1.02)
plt.tight_layout()
plt.show()

print("""Key insight:
- If all CATEs are similar (left), a simple ATE is enough — no need for a causal forest.
- If CATEs are highly dispersed (right), a targeting policy can dramatically outperform
  universal nudging by concentrating effort on the most responsive customers.
- The causal forest's value is proportional to the spread of the CATE distribution.
""")

---
## 8.3 Cross-Industry Extensions

### Academic Discussion

The environmental framing of von Zahn et al. (2024) positions product-return nudging within the broader literature on **pro-environmental behaviour change** (Thaler & Sunstein, 2008; Allcott, 2011) and **sustainable operations** (Souza, 2013). However, the methodological pipeline — RCT design, digital footprint features, causal ML targeting — is transferable to any setting where (a) a firm or institution can randomise a low-cost digital intervention, (b) individual-level behavioural data is available pre-treatment, and (c) the intervention's effect is plausibly heterogeneous across individuals. Three extensions are of particular academic and practical interest. First, **multi-armed settings** where several nudge variants (e.g., different message framings) compete for a single customer impression; here the framework extends naturally to the Causal Multi-Armed Bandit literature (Luedtke & van der Laan, 2016). Second, **dynamic targeting** where CATE estimates are updated sequentially as new purchase data arrives — an online learning problem solvable via Thompson sampling over the causal forest's posterior. Third, **spillover and network effects**: if nudged customers share information (e.g., social media posts about sustainability), the SUTVA assumption underlying the current ATE estimate is violated, and partial identification bounds or spatial/network IV methods become necessary (Manski, 2013).

**Plain-language summary:**
- This exact approach — identify who responds, only nudge them, measure the profit — can be used in many industries beyond fashion e-commerce.
- Three interesting directions to extend the research: (1) test *multiple* different nudge messages at once and learn which works best for each customer type; (2) update your targeting model continuously as new data comes in, rather than training once; (3) account for the fact that one nudged customer might tell their friends, meaning the effect spreads beyond who you directly targeted.
- The simulations below illustrate the first two of these extensions with toy examples.

In [ ]:
# ── Extension 8.3a: Multi-arm illustration ────────────────────────────────────
# Suppose there are 3 nudge variants: environmental message, social norm message,
# and financial incentive. Each arm has a different CATE distribution.
# This illustrates why estimating per-arm CATEs matters for targeting.

rng = np.random.default_rng(RANDOM_SEED)
n_sim = 3_000

# Simulate customer features
past_return_rate  = rng.beta(2, 5, n_sim)          # skewed toward low returners
env_sensitivity   = rng.uniform(0, 1, n_sim)        # how green-minded is customer
price_sensitivity = rng.uniform(0, 1, n_sim)        # how price-conscious

# Simulate arm-specific CATEs
# Environmental message: works best for green-minded, high-returner customers
cate_env  = -0.02 - 0.10 * env_sensitivity * past_return_rate + rng.normal(0, 0.02, n_sim)

# Social norm: works broadly but weakly
cate_norm = -0.03 - 0.04 * past_return_rate + rng.normal(0, 0.02, n_sim)

# Financial incentive: works best for price-sensitive, high-returner customers
cate_fin  = -0.01 - 0.12 * price_sensitivity * past_return_rate + rng.normal(0, 0.02, n_sim)

# Best arm per customer (most negative CATE wins)
cate_stack = np.stack([cate_env, cate_norm, cate_fin], axis=1)
best_arm   = np.argmin(cate_stack, axis=1)
best_cate  = cate_stack[np.arange(n_sim), best_arm]

arm_names  = ['Environmental', 'Social Norm', 'Financial']
arm_colors = ['#4CAF82', '#5B8DB8', '#E07B54']

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# CATE distributions by arm
for cate_arr, name, color in zip([cate_env, cate_norm, cate_fin], arm_names, arm_colors):
    axes[0].hist(cate_arr, bins=50, alpha=0.5, label=f'{name} (mean={cate_arr.mean():.3f})',
                 color=color, edgecolor='white')
axes[0].axvline(0, color='black', linestyle='--', linewidth=1, alpha=0.6)
axes[0].set_title('CATE Distributions by Nudge Arm')
axes[0].set_xlabel('CATE')
axes[0].set_ylabel('Count')
axes[0].legend(fontsize=8)
axes[0].xaxis.set_major_formatter(mtick.PercentFormatter(1.0, decimals=1))

# Share assigned to each arm under personalised routing
arm_shares = pd.Series(best_arm).value_counts().sort_index()
arm_shares.index = arm_names
arm_shares.plot(kind='bar', ax=axes[1], color=arm_colors, edgecolor='white')
axes[1].set_title('Optimal Arm Assignment\n(personalised routing)')
axes[1].set_xlabel('')
axes[1].set_ylabel('N customers')
axes[1].set_xticklabels(arm_names, rotation=0)

plt.tight_layout()
plt.show()

print(f"Mean CATE — best arm per customer  : {best_cate.mean():+.4f}")
print(f"Mean CATE — always environmental   : {cate_env.mean():+.4f}")
print(f"Mean CATE — always social norm     : {cate_norm.mean():+.4f}")
print(f"Mean CATE — always financial       : {cate_fin.mean():+.4f}")
print(f"\nGain from personalised arm routing vs. best single arm: "
      f"{best_cate.mean() - min(cate_env.mean(), cate_norm.mean(), cate_fin.mean()):+.4f}")

In [ ]:
# ── Extension 8.3b: Dynamic targeting — value of model updating ───────────────
# Simulates what happens to targeting performance as the causal forest is
# retrained on growing amounts of data over time (monthly batches).
# In practice: the more data you accumulate, the tighter the CATE estimates
# and the better the targeting policy.

rng = np.random.default_rng(RANDOM_SEED)

# Simulate: true CATE is unknown; our estimate improves with sample size
# Proxy: CATE estimation error ~ 1/sqrt(n)
sample_sizes    = [200, 500, 1_000, 2_000, 5_000, 10_000, 20_000]
true_cate_std   = 0.07    # from our replication
true_cate_mean  = -0.05

results_dynamic = []
for n_obs in sample_sizes:
    # Simulate CATE estimation error scaling as 1/sqrt(n)
    estimation_noise = 1 / np.sqrt(n_obs)
    true_cates  = rng.normal(true_cate_mean, true_cate_std, 5_000)   # fixed test set
    est_cates   = true_cates + rng.normal(0, estimation_noise * true_cate_std, 5_000)

    # Policy: nudge if estimated CATE < 0
    nudge_mask_smart = est_cates < 0
    nudge_mask_univ  = np.ones(5_000, dtype=bool)   # nudge everyone

    # True profit (using true CATEs, not estimated)
    profit_smart = (-true_cates[nudge_mask_smart].clip(max=0)).sum() * 15 \
                   - nudge_mask_smart.sum() * 0.10
    profit_univ  = (-true_cates.clip(max=0)).sum() * 15 - 5_000 * 0.10
    results_dynamic.append({
        'n_training': n_obs,
        'profit_smart': profit_smart,
        'profit_univ':  profit_univ,
        'personalisation_value': profit_smart - profit_univ,
        'estimation_noise': estimation_noise,
    })

dyn_df = pd.DataFrame(results_dynamic)

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(dyn_df['n_training'], dyn_df['profit_smart'], 'o-',
        color='#4CAF82', linewidth=2.5, markersize=7, label='Smart targeting profit')
ax.plot(dyn_df['n_training'], dyn_df['profit_univ'],  's--',
        color='#5B8DB8', linewidth=2,   markersize=7, label='Universal nudging profit')
ax.set_xscale('log')
ax.set_xlabel('Training sample size (log scale)')
ax.set_ylabel('Profit gain (€, on fixed test set of 5,000)')
ax.set_title('Dynamic Targeting: How Targeting Quality Improves with More Training Data')
ax.legend()
plt.tight_layout()
plt.show()

print("\nWith small samples, noisy CATE estimates mean smart targeting may not beat universal.")
print("As data accumulates, the value of personalisation grows.")
print(dyn_df[['n_training','profit_smart','profit_univ','personalisation_value']].round(2).to_string(index=False))

---
## References

- von Zahn, M., Feuerriegel, S., & Kuehl, N. (2024). **Smart Green Nudging: Reducing Product Returns Through Digital Footprints and Causal Machine Learning.** *Marketing Science*. https://doi.org/10.1287/mksc.2022.0393
- Wager, S., & Athey, S. (2018). Estimation and inference of heterogeneous treatment effects using random forests. *JASA*, 113(523), 1228–1242.
- Chernozhukov, V., Chetverikov, D., Demirer, M., Duflo, E., Hansen, C., Newey, W., & Robins, J. (2018). Double/debiased machine learning for treatment and structural parameters. *The Econometrics Journal*, 21(1), C1–C68.
- Athey, S., & Wager, S. (2021). Policy learning with observational data. *Econometrica*, 89(1), 133–161.
- Allcott, H. (2011). Social norms and energy conservation. *Journal of Public Economics*, 95(9–10), 1082–1095.
- Thaler, R. H., & Sunstein, C. R. (2008). *Nudge: Improving Decisions about Health, Wealth, and Happiness*. Yale University Press.
- Manski, C. F. (2013). Identification of treatment response with social interactions. *The Econometrics Journal*, 16(1), S1–S23.
- EconML: https://econml.azurewebsites.net/